In [0]:
import mlflow

mlflow.set_experiment(experiment_id="522185415005810")

# Define a new agent version, represented as an MLflow LoggedModel
mlflow.set_active_model(name="my-agent-v1")

# Log LLM hyperparameters, prompts, and more
mlflow.log_model_params({
    "prompt_template": "My prompt template",
    "llm": "databricks-llama-4-maverick",
    "temperature": 0.2,
})

# Define agent code and add MLflow tracing to capture requests and responses.
# (Replace this with your agent code)
@mlflow.trace
def predict(query):
    return f"Response to query: {query}"

# Run your agent code. Resulting traces are automatically linked to
# your agent version.
predict("What is MLflow?")

## Build a real agent from `my-agent-v1`
The cells below turn the `my-agent-v1` agent version into a working MLflow `ResponsesAgent`:
1. Load the LLM, temperature, and prompt template stored as params on `my-agent-v1`.
2. Define an agent that calls that LLM through Databricks Model Serving.
3. Test it locally (traces link to `my-agent-v1`).
4. Log the agent code into the `my-agent-v1` LoggedModel and validate the logged model.

In [0]:
import mlflow

mlflow.set_experiment(experiment_id="522185415005810")

# Reuse the existing agent version so new traces and artifacts attach to it
active_model = mlflow.set_active_model(name="my-agent-v1")
agent_version = mlflow.get_logged_model(active_model.model_id)
params = agent_version.params

LLM_ENDPOINT = params["llm"]
TEMPERATURE = float(params["temperature"])
# The stored prompt template is a placeholder; fall back to a sensible system prompt
SYSTEM_PROMPT = (
    params["prompt_template"]
    if params["prompt_template"] != "My prompt template"
    else "You are a helpful, concise assistant. Answer clearly and say when you don't know."
)

print(f"Agent version: {agent_version.name} ({agent_version.model_id})")
print(f"LLM: {LLM_ENDPOINT} | temperature: {TEMPERATURE}")
print(f"System prompt: {SYSTEM_PROMPT}")

In [0]:
import uuid
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse


class MyAgent(ResponsesAgent):
    """Chat agent that calls a Databricks-served LLM with the my-agent-v1 settings."""

    def __init__(self, llm_endpoint: str, temperature: float, system_prompt: str):
        self.llm_endpoint = llm_endpoint
        self.temperature = temperature
        self.system_prompt = system_prompt

    @mlflow.trace(span_type=SpanType.LLM)
    def call_llm(self, messages: list[dict]) -> str:
        w = WorkspaceClient()
        resp = w.api_client.do(
            "POST",
            f"/serving-endpoints/{self.llm_endpoint}/invocations",
            body={"messages": messages, "temperature": self.temperature},
        )
        return resp["choices"][0]["message"]["content"]

    # predict is traced automatically for ResponsesAgent subclasses
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        messages = [{"role": "system", "content": self.system_prompt}]
        messages += self.prep_msgs_for_cc_llm([m.model_dump() for m in request.input])
        answer = self.call_llm(messages)
        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text=answer, id=str(uuid.uuid4()))]
        )


agent = MyAgent(LLM_ENDPOINT, TEMPERATURE, SYSTEM_PROMPT)

In [0]:
test_request = ResponsesAgentRequest(input=[{"role": "user", "content": "What is MLflow?"}])
response = agent.predict(test_request)
print(response.output[0].content[0]["text"])

In [0]:
from mlflow.models.resources import DatabricksServingEndpoint

input_example = {"input": [{"role": "user", "content": "What is MLflow?"}]}

logged_agent = mlflow.pyfunc.log_model(
    name="my-agent-v1",
    model_id=agent_version.model_id,  # write into the existing my-agent-v1 version
    python_model=agent,
    input_example=input_example,
    resources=[DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT)],
    pip_requirements=[f"mlflow=={mlflow.__version__}", "databricks-sdk"],
)
print(f"Logged model URI: {logged_agent.model_uri}")

In [0]:
loaded_agent = mlflow.pyfunc.load_model(logged_agent.model_uri)
result = loaded_agent.predict(
    {"input": [{"role": "user", "content": "Give me one tip for writing good prompts."}]}
)
print(result["output"][0]["content"][0]["text"])